# CreditLens — Day 5 XGBoost review

This executed notebook reads the saved Day 5 artefacts. It does not retrain, rescore validation, load final-test outcomes, select a threshold or perform SHAP analysis.

In [1]:
from pathlib import Path
import csv
import json

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'reports').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
metrics = json.loads((PROJECT_ROOT / 'reports/day5_xgboost_validation_metrics.json').read_text())
selected = json.loads((PROJECT_ROOT / 'reports/day5_xgboost_selected_params.json').read_text())
with (PROJECT_ROOT / 'reports/day5_validation_model_comparison.csv').open(newline='') as source:
    comparison = list(csv.DictReader(source))

## Training-only grouped search

In [2]:
print(f"XGBoost version: {selected['xgboost_version']}")
print(f"Configurations: {selected['search_iterations']}")
print(f"CV fits: {selected['fitted_candidates']}")
print(f"Search seconds: {selected['search_seconds']:.3f}")
print(f"Best grouped-CV AP: {selected['best_cv_average_precision']:.6f}")
print(f"Best configuration grouped-CV ROC-AUC: {selected['best_cv_roc_auc']:.6f}")
print(f"Maximum fold group overlap: {max(fold['overlapping_groups'] for fold in selected['cv_group_audit'])}")
selected['selected_hyperparameters']

XGBoost version: 2.1.4
Configurations: 12
CV fits: 48
Search seconds: 669.438
Best grouped-CV AP: 0.562699
Best configuration grouped-CV ROC-AUC: 0.786319
Maximum fold group overlap: 0


{'colsample_bytree': 0.85,
 'gamma': 0.0,
 'learning_rate': 0.03,
 'max_depth': 3,
 'min_child_weight': 1,
 'n_estimators': 600,
 'reg_lambda': 10.0,
 'scale_pos_weight': 3.5,
 'subsample': 0.85}

## Validation comparison

Average Precision is the primary ranking comparison and ROC-AUC is secondary. Threshold 0.5 is diagnostic only. The numerical comparison does not select a final model.

In [3]:
comparison

[{'model': 'Logistic Regression',
  'roc_auc': '0.7544054889873056',
  'average_precision': '0.519470900444617',
  'precision_at_0_5': '0.6696165191740413',
  'recall_at_0_5': '0.3421250941974378',
  'f1_at_0_5': '0.45286783042394013',
  'proportion_flagged_at_0_5': '0.113'},
 {'model': 'Random Forest',
  'roc_auc': '0.77191665762253',
  'average_precision': '0.539979526640148',
  'precision_at_0_5': '0.6618287373004355',
  'recall_at_0_5': '0.3436322532027129',
  'f1_at_0_5': '0.4523809523809524',
  'proportion_flagged_at_0_5': '0.11483333333333333'},
 {'model': 'XGBoost',
  'roc_auc': '0.7717920823677071',
  'average_precision': '0.5366433171661142',
  'precision_at_0_5': '0.4515951595159516',
  'recall_at_0_5': '0.6186887716654107',
  'f1_at_0_5': '0.5220985691573927',
  'proportion_flagged_at_0_5': '0.303'}]

## XGBoost validation curves

![ROC](../reports/figures/day5_xgboost_validation_roc.png)

![Precision–recall](../reports/figures/day5_xgboost_validation_precision_recall.png)

## Gain importance

Gain is associative and model-specific. Correlated predictors can split or mask importance, and the values are not causal.

In [4]:
with (PROJECT_ROOT / 'reports/day5_xgboost_source_importance.csv').open(newline='') as source:
    source_importance = list(csv.DictReader(source))
source_importance[:10]

[{'source_feature': 'PAY_2',
  'aggregated_gain': '662.375097752',
  'normalised_gain': '0.321279847481'},
 {'source_feature': 'PAY_0',
  'aggregated_gain': '474.948054314',
  'normalised_gain': '0.230369829677'},
 {'source_feature': 'PAY_3',
  'aggregated_gain': '250.319054127',
  'normalised_gain': '0.121415294452'},
 {'source_feature': 'PAY_5',
  'aggregated_gain': '174.057940006',
  'normalised_gain': '0.0844254390116'},
 {'source_feature': 'PAY_4',
  'aggregated_gain': '164.721596718',
  'normalised_gain': '0.0798969189058'},
 {'source_feature': 'PAY_6',
  'aggregated_gain': '105.046440482',
  'normalised_gain': '0.0509519522866'},
 {'source_feature': 'PAY_AMT1',
  'aggregated_gain': '38.41771698',
  'normalised_gain': '0.0186342123878'},
 {'source_feature': 'PAY_AMT3',
  'aggregated_gain': '25.8236579895',
  'normalised_gain': '0.0125255628245'},
 {'source_feature': 'PAY_AMT2',
  'aggregated_gain': '25.5145683289',
  'normalised_gain': '0.0123756413082'},
 {'source_feature': 'LIM

In [5]:
{
    'validation_used_for_selection': metrics['validation_used_for_hyperparameter_selection'],
    'test_partition_evaluated': metrics['test_partition_evaluated'],
    'threshold_selected': metrics['threshold_selected'],
    'threshold_status': metrics['threshold_status'],
    'search_warnings': metrics['search_warnings'],
}

{'validation_used_for_selection': False,
 'test_partition_evaluated': False,
 'threshold_selected': False,
 'threshold_status': 'Baseline reporting only; not an operating threshold.',
 'search_warnings': []}